# xT Model

## Imports

In [1]:
import sys
print(sys.executable)
import numpy as np
import pandas as pd
import psycopg2
import matplotlib.patheffects as path_effects
import matplotlib.pyplot as plt
from mplsoccer import Sbopen, Pitch

parser = Sbopen()
pitch = Pitch(line_zorder=2, pitch_type='opta')

c:\Users\rogel\AppData\Local\Programs\Python\Python39\python.exe


In [2]:
from pathlib import Path
import sys
import joblib

# Define project root (two levels up from the xT folder)
ROOT = Path.cwd().parent.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Import helper functions
from Scripts.utils import calculate_xg_xgot

# Load trained models
xg_model = joblib.load(ROOT / "Models" / "xG" / "xg_model.pkl")
xgot_model = joblib.load(ROOT / "Models" / "xGOT" / "xgot_model.pkl")

print(f"Imported calculate_xg_xgot and loaded models, ROOT={ROOT}")

Imported calculate_xg_xgot and loaded models, ROOT=c:\Users\rogel\Desktop\Segunda Tesina


## Get the Data

In [3]:

conn = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    password="admin",
    host="localhost",
    port=5432
)

In [4]:
cursor = conn.cursor()

In [ ]:
cursor.execute("""
Select *
FROM match_events
WHERE 1=1
AND type_display_name IN ('BlockedPass','Clearance','MissedShots','GoodSkill','Goal','Pass','OffsidePass','TakeOn','SavedShot')
""")

KeyboardInterrupt: 

: 

In [ ]:
records = cursor.fetchall()

In [ ]:
columns = [desc[0] for desc in cursor.description]

In [ ]:
df_match = pd.DataFrame(records, columns=columns)

In [ ]:
df_match.shape

In [ ]:
df_match = calculate_xg_xgot(df_match,xg_model,xgot_model)

In [ ]:
df_match.columns

In [ ]:
df_match['outcome_type_display_name'].value_counts()

## Filter the Data

In [ ]:
bins = (32, 24)

In [ ]:
# boolean columns for working out probabilities
df_match['goal'] = df_match['type_display_name'] == 'Goal'
df_match['shoot'] = df_match['type_display_name'].isin(['MissedShots', 'SavedShot'])
df_match['move'] = df_match['type_display_name'].isin([
    'BlockedPass', 'Clearance', 'GoodSkill',
    'Pass', 'OffsidePass', 'TakeOn'
])


In [ ]:
shot_probability = pitch.bin_statistic(df_match['x'], df_match['y'], values=df_match['shoot'],
                                       statistic='mean', bins=bins)
move_probability = pitch.bin_statistic(df_match['x'], df_match['y'], values=df_match['move'],
                                       statistic='mean', bins=bins)
goal_probability = pitch.bin_statistic(df_match.loc[df_match['shoot'], 'x'],
                                       df_match.loc[df_match['shoot'], 'y'],
                                       df_match.loc[df_match['shoot'], 'xG_pred'],
                                       statistic='mean',
                                       bins=bins)

In [ ]:
fig, ax = pitch.draw()
shot_heatmap = pitch.heatmap(shot_probability, ax=ax)

In [ ]:
fig, ax = pitch.draw()
move_heatmap = pitch.heatmap(move_probability, ax=ax)

In [ ]:
fig, ax = pitch.draw()
goal_heatmap = pitch.heatmap(goal_probability, ax=ax)

In [ ]:
# get a dataframe of move events and filter it
# so the dataframe only contains actions inside the pitch.
print (1)
move = df_match[df_match['move']].copy()
bin_start_locations = pitch.bin_statistic(move['x'], move['y'], bins=bins)
move = move[bin_start_locations['inside']].copy()
print(2)
# get the successful moves, which filters out the events that ended outside the pitch
# or where not successful (null)
bin_end_locations = pitch.bin_statistic(move['end_x'], move['end_y'], bins=bins)
move_success = move[(bin_end_locations['inside']) & (move['outcome_type_display_name']=='Successful')].copy()
print(3)
# get a dataframe of the successful moves
# and the grid cells they started and ended in
bin_success_start = pitch.bin_statistic(move_success['x'], move_success['y'], bins=bins)
bin_success_end = pitch.bin_statistic(move_success['end_x'], move_success['end_y'], bins=bins)
df_bin = pd.DataFrame({'x': bin_success_start['binnumber'][0],
                       'y': bin_success_start['binnumber'][1],
                       'end_x': bin_success_end['binnumber'][0],
                       'end_y': bin_success_end['binnumber'][1]})
print(4)
# calculate the bin counts for the successful moves, i.e. the number of moves between grid cells
bin_counts = df_bin.value_counts().reset_index(name='bin_counts')
print(5)
# create the move_transition_matrix of shape (num_y_bins, num_x_bins, num_y_bins, num_x_bins)
# this is the number of successful moves between grid cells.
num_y, num_x = shot_probability['statistic'].shape
move_transition_matrix = np.zeros((num_y, num_x, num_y, num_x))
move_transition_matrix[bin_counts['y'], bin_counts['x'],
                       bin_counts['end_y'], bin_counts['end_x']] = bin_counts.bin_counts.values
print(6)
# and divide by the starting locations for all moves (including unsuccessful)
# to get the probability of moving the ball successfully between grid cells
bin_start_locations = pitch.bin_statistic(move['x'], move['y'], bins=bins)
bin_start_locations = np.expand_dims(bin_start_locations['statistic'], (2, 3))
move_transition_matrix = np.divide(move_transition_matrix,
                                   bin_start_locations,
                                   out=np.zeros_like(move_transition_matrix),
                                   where=bin_start_locations != 0,
                                   )

## Model

In [ ]:
move_transition_matrix = np.nan_to_num(move_transition_matrix)
shot_probability_matrix = np.nan_to_num(shot_probability['statistic'])
move_probability_matrix = np.nan_to_num(move_probability['statistic'])
goal_probability_matrix = np.nan_to_num(goal_probability['statistic'])

In [ ]:
xt = np.multiply(shot_probability_matrix, goal_probability_matrix)
diff = 1
iteration = 0
while np.any(diff > 0.00001):  # iterate until the differences between the old and new xT is small
    xt_copy = xt.copy()  # keep a copy for comparing the differences
    # calculate the new expected threat
    xt = (np.multiply(shot_probability_matrix, goal_probability_matrix) +
          np.multiply(move_probability_matrix,
                      np.multiply(move_transition_matrix, np.expand_dims(xt, axis=(0, 1))).sum(
                          axis=(2, 3)))
          )
    diff = (xt - xt_copy)
    iteration += 1
print('Number of iterations:', iteration)

In [ ]:
path_eff = [path_effects.Stroke(linewidth=1.5, foreground='black'),
            path_effects.Normal()]
# new bin statistic for plotting xt only
for_plotting = pitch.bin_statistic(df_match['x'], df_match['y'], bins=bins)
for_plotting['statistic'] = xt
fig, ax = pitch.draw(figsize=(14, 9.625))
_ = pitch.heatmap(for_plotting, ax=ax)
# sphinx_gallery_thumbnail_path = 'gallery/tutorials/images/sphx_glr_plot_xt_improvements_004'

In [ ]:
xt.shape

In [ ]:
import numpy as np

nrows = xt.shape[0]

for i in range(nrows // 2):
    top = xt[i, :]
    bottom = xt[-i-1, :]
    m = np.minimum(top, bottom)
    xt[i, :] = m
    xt[-i-1, :] = m


In [ ]:
path_eff = [path_effects.Stroke(linewidth=1.5, foreground='black'),
            path_effects.Normal()]
# new bin statistic for plotting xt only
for_plotting = pitch.bin_statistic(df_match['x'], df_match['y'], bins=bins)
for_plotting['statistic'] = xt
fig, ax = pitch.draw(figsize=(14, 9.625))
_ = pitch.heatmap(for_plotting, ax=ax)
# sphinx_gallery_thumbnail_path = 'gallery/tutorials/images/sphx_glr_plot_xt_improvements_004'

## Export Grid

In [ ]:
# first get grid start and end cells
grid_start = pitch.bin_statistic(move_success.x, move_success.y, bins=bins)
grid_end = pitch.bin_statistic(move_success.end_x, move_success.end_y, bins=bins)

# then get the xT values from the start and end grid cell
start_xt = xt[grid_start['binnumber'][1], grid_start['binnumber'][0]]
end_xt = xt[grid_end['binnumber'][1], grid_end['binnumber'][0]]

# then calculate the added xT
added_xt = end_xt - start_xt
move_success['xt'] = added_xt

In [ ]:
xt.shape

In [ ]:
xtdf = pd.DataFrame(xt)
xtdf.to_csv("xt.csv", index=False, header=False)

In [ ]:
xtdf